# Análisis Exploratorio de Datos (EDA)
## Stack Overflow Developer Survey 2023

**Dataset:** Stack Overflow Annual Developer Survey 2023
**Fuente:** [Kaggle – stackoverflow/stack-overflow-developer-survey](https://www.kaggle.com/datasets/stackoverflow/stack-overflow-developer-survey)
**Filas:** ~89.184 respuestas  |  **Columnas seleccionadas:** 13

---

## 0. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Configuración global de visualizaciones
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"]      = 12

print("Librerías importadas correctamente.")

---
## 1. Selección y carga del dataset

El dataset proviene de la encuesta anual de Stack Overflow 2023, respondida por
~89 000 desarrolladores de todo el mundo. Contiene información sobre salarios,
experiencia, educación, modalidad de trabajo y herramientas utilizadas.

> **Instrucción:** colocar `survey_results_public.csv` en el mismo directorio que este notebook.

In [ ]:
# ── Carga ──────────────────────────────────────────────────────────────────
df_raw = pd.read_csv("survey_results_public.csv")

print(f"Shape del dataset original : {df_raw.shape}")
print(f"  Filas   : {df_raw.shape[0]:,}")
print(f"  Columnas: {df_raw.shape[1]}")
df_raw.head(3)

In [ ]:
# ── Selección de columnas relevantes ───────────────────────────────────────
COLS = [
    "ResponseId",           # Identificador único
    "Age",                  # Rango etario del respondente
    "Country",              # País de residencia
    "EdLevel",              # Nivel educativo más alto alcanzado
    "Employment",           # Situación laboral (puede ser multi-valor)
    "RemoteWork",           # Modalidad de trabajo
    "YearsCodePro",         # Años de experiencia profesional programando
    "WorkExp",              # Años de experiencia laboral total
    "ConvertedCompYearly",  # Salario anual en USD (normalizado por PPP)
    "DevType",              # Tipo de desarrollador (puede ser multi-valor)
    "OrgSize",              # Tamaño de la organización
    "JobSat",               # Satisfacción laboral
    "AISelect",             # Uso de herramientas de IA en el trabajo
]

df = df_raw[COLS].copy()
print(f"Shape tras selección de columnas: {df.shape}")
df.dtypes

---
## 2. Hipótesis planteadas

A partir del contexto del dataset se formularon las siguientes cinco hipótesis:

| # | Hipótesis |
|---|-----------|
| H1 | Los desarrolladores con **más años de experiencia profesional** tienen salarios significativamente más altos. |
| H2 | El trabajo **remoto** está asociado a salarios más altos que el trabajo presencial. |
| H3 | El **nivel educativo formal** influye positivamente en el salario. |
| H4 | Los desarrolladores en **empresas más grandes** ganan más que los de empresas pequeñas. |
| H5 | Existen **diferencias salariales significativas** según el tipo de desarrollador (role). |

---
## 3. Limpieza y preparación de datos

### 3.1 Valores nulos

In [ ]:
nulos   = df.isnull().sum()
pct     = (nulos / len(df) * 100).round(2)
resumen = pd.DataFrame({"Nulos": nulos, "Porcentaje (%)": pct})
resumen = resumen[resumen["Nulos"] > 0].sort_values("Porcentaje (%)", ascending=False)

print("Columnas con valores nulos:")
resumen

**Interpretación:**
- `ConvertedCompYearly` tiene alto % de nulos porque muchos encuestados no reportan
  su salario (desempleados, estudiantes, o quienes prefieren no revelar ingresos).
  Se trabajará con un subconjunto `df_sal` únicamente para los análisis salariales.
- `RemoteWork`, `OrgSize` y `DevType` tienen nulos moderados; se omitirán puntualmente
  en los gráficos que los requieran.

### 3.2 Duplicados

In [ ]:
n_dup = df.duplicated().sum()
print(f"Filas duplicadas: {n_dup}")
# Cada fila corresponde a una respuesta única identificada por ResponseId
print(f"ResponseId únicos: {df['ResponseId'].nunique():,}  (= total filas → sin duplicados)")

### 3.3 Conversión de experiencia a numérico

In [ ]:
def limpiar_anios(valor):
    """Convierte strings de años de experiencia al tipo float.

    Casos especiales del dataset:
        'Less than 1 year'  -> 0.5
        'More than 50 years' -> 50.0
    """
    if pd.isna(valor):
        return np.nan
    if valor == "Less than 1 year":
        return 0.5
    if valor == "More than 50 years":
        return 50.0
    try:
        return float(valor)
    except ValueError:
        return np.nan

df["YearsCodePro"] = df["YearsCodePro"].apply(limpiar_anios)
df["WorkExp"]      = df["WorkExp"].apply(limpiar_anios)

print("YearsCodePro — estadísticas tras limpieza:")
print(df["YearsCodePro"].describe().apply(lambda x: f"{x:.2f}"))
print()
print("WorkExp — estadísticas tras limpieza:")
print(df["WorkExp"].describe().apply(lambda x: f"{x:.2f}"))

### 3.4 Simplificación de DevType (campo multi-valor)

In [ ]:
def simplificar_devtype(valor):
    """Asigna un único rol principal a partir de la cadena multi-valor de DevType."""
    if pd.isna(valor):
        return np.nan
    v = valor.lower()
    if "data scientist" in v or "machine learning" in v:
        return "Data Scientist / ML"
    if "data" in v and "analyst" in v:
        return "Data Analyst"
    if "full-stack" in v:
        return "Full-stack Developer"
    if "back-end" in v:
        return "Back-end Developer"
    if "front-end" in v:
        return "Front-end Developer"
    if "devops" in v or "site reliability" in v:
        return "DevOps / SRE"
    if "mobile" in v:
        return "Mobile Developer"
    if "embedded" in v:
        return "Embedded Developer"
    if "game" in v:
        return "Game Developer"
    if "security" in v:
        return "Security Engineer"
    if "engineering manager" in v:
        return "Engineering Manager"
    if "student" in v:
        return "Student"
    return "Other"

df["DevType_simp"] = df["DevType"].apply(simplificar_devtype)
print("Distribución de roles simplificados:")
print(df["DevType_simp"].value_counts().to_string())

### 3.5 Etiquetas cortas para EdLevel y OrgSize

In [ ]:
# ── EdLevel ─────────────────────────────────────────────────────────────────
EDLEVEL_MAP = {
    "Bachelor's degree (B.A., B.S., B.Eng., etc.)":                              "Licenciatura",
    "Master's degree (M.A., M.S., M.Eng., MBA, etc.)":                           "Maestría",
    "Some college/university study without earning a degree":                      "Univ. sin título",
    "Associate degree (A.A., A.S., etc.)":                                        "Tecnicatura",
    "Primary/elementary school":                                                   "Primaria",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "Secundaria",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)":                            "Doctorado / PhD",
    "Something else":                                                              "Otro",
}
df["EdLevel_short"] = df["EdLevel"].map(EDLEVEL_MAP).fillna(df["EdLevel"])

# Orden lógico (de menor a mayor nivel educativo)
EDLEVEL_ORDER = [
    "Primaria", "Secundaria", "Univ. sin título",
    "Tecnicatura", "Licenciatura", "Maestría", "Doctorado / PhD",
]

# ── OrgSize ──────────────────────────────────────────────────────────────────
ORGSIZE_MAP = {
    "Just me - I am a one-man shop": "Solo",
    "2 to 9 employees":              "2-9",
    "10 to 19 employees":            "10-19",
    "20 to 99 employees":            "20-99",
    "100 to 499 employees":          "100-499",
    "500 to 999 employees":          "500-999",
    "1,000 to 4,999 employees":      "1K-5K",
    "5,000 to 9,999 employees":      "5K-10K",
    "10,000 or more employees":      "+10K",
    "I don't know":                  "No sabe",
}
ORGSIZE_ORDER = ["Solo","2-9","10-19","20-99","100-499","500-999","1K-5K","5K-10K","+10K"]

df["OrgSize_short"] = df["OrgSize"].map(ORGSIZE_MAP).fillna(df["OrgSize"])
df["OrgSize_short"] = pd.Categorical(df["OrgSize_short"], categories=ORGSIZE_ORDER, ordered=True)

print("EdLevel_short — valores únicos:")
print(df["EdLevel_short"].value_counts().to_string())

### 3.6 Subconjunto salarial — df_sal

In [ ]:
# Nos quedamos solo con respuestas que tienen salario reportado y razonable
df_sal = df[df["ConvertedCompYearly"].notna()].copy()
df_sal = df_sal[df_sal["ConvertedCompYearly"] > 1_000]       # Descartamos salarios casi nulos

# Recortamos el 1 % superior para evitar outliers extremos en visualizaciones
p99 = df_sal["ConvertedCompYearly"].quantile(0.99)
df_sal = df_sal[df_sal["ConvertedCompYearly"] <= p99]

print(f"Filas con salario válido : {len(df_sal):,}")
print(f"Salario mínimo  : USD {df_sal['ConvertedCompYearly'].min():>12,.0f}")
print(f"Salario mediano : USD {df_sal['ConvertedCompYearly'].median():>12,.0f}")
print(f"Salario medio   : USD {df_sal['ConvertedCompYearly'].mean():>12,.0f}")
print(f"Salario máximo  : USD {df_sal['ConvertedCompYearly'].max():>12,.0f}")
print(f"\nUmbral 99 %     : USD {p99:>12,.0f}  (filas eliminadas como outliers extremos)")

### 3.7 Resumen del dataset limpio

In [ ]:
print("=== Dataset completo (df) ===")
print(f"  Shape : {df.shape}")
print()
print("Tipos de datos finales:")
print(df.dtypes)
print()
print("=== Subconjunto salarial (df_sal) ===")
print(f"  Shape : {df_sal.shape}")

---
## 4. Análisis Exploratorio de Datos (EDA)

### 4.1 Tipos de variables y estadísticas descriptivas

In [ ]:
print("── Variables NUMÉRICAS ──────────────────────────────────────")
cols_num = ["ConvertedCompYearly", "YearsCodePro", "WorkExp"]
df_sal[cols_num].describe().T.style.format("{:.2f}")

In [ ]:
print("── Variables CATEGÓRICAS (top valores) ─────────────────────")
for col in ["RemoteWork", "EdLevel_short", "DevType_simp", "OrgSize_short", "AISelect"]:
    print(f"\n{col}:")
    print(df[col].value_counts().head(5).to_string())

---
### 4.2 Distribuciones — Histogramas con KDE

> Se grafican las tres variables numéricas continuas principales.

In [ ]:
# ── Histograma 1: Salario anual (USD) ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

sns.histplot(
    data=df_sal, x="ConvertedCompYearly",
    bins=70, kde=True, color="steelblue", alpha=0.75, ax=ax
)

mediana = df_sal["ConvertedCompYearly"].median()
media   = df_sal["ConvertedCompYearly"].mean()
ax.axvline(mediana, color="crimson",     ls="--", lw=2, label=f"Mediana: USD {mediana:,.0f}")
ax.axvline(media,   color="darkorange",  ls="--", lw=2, label=f"Media:   USD {media:,.0f}")

ax.set_title("Distribución del Salario Anual (USD)", fontsize=15, fontweight="bold")
ax.set_xlabel("Salario anual (USD)")
ax.set_ylabel("Frecuencia")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${{x/1000:.0f}}K"))
ax.legend()
plt.tight_layout()
plt.savefig("hist_salario.png", dpi=150, bbox_inches="tight")
plt.show()

skew = df_sal["ConvertedCompYearly"].skew()
print(f"Asimetría (skewness): {skew:.2f}")
print("→ Distribución asimétrica positiva: la mayoría gana menos que la media.")
print("  Los salarios muy altos 'tiran' la media hacia arriba.")

In [ ]:
# ── Histograma 2: Años de experiencia profesional ────────────────────────────
df_exp = df.dropna(subset=["YearsCodePro"])

fig, ax = plt.subplots(figsize=(12, 5))

sns.histplot(
    data=df_exp, x="YearsCodePro",
    bins=40, kde=True, color="mediumseagreen", alpha=0.75, ax=ax
)

med_exp = df_exp["YearsCodePro"].median()
ax.axvline(med_exp, color="crimson", ls="--", lw=2, label=f"Mediana: {med_exp:.0f} años")

ax.set_title("Distribución de Años de Experiencia Profesional Programando", fontsize=15, fontweight="bold")
ax.set_xlabel("Años de experiencia profesional")
ax.set_ylabel("Frecuencia")
ax.legend()
plt.tight_layout()
plt.savefig("hist_exp_pro.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mediana: {med_exp:.1f} años  |  Media: {df_exp['YearsCodePro'].mean():.1f} años")
print(f"Asimetría: {df_exp['YearsCodePro'].skew():.2f}  → sesgo positivo: la mayoría tiene menos de 10 años de experiencia.")

In [ ]:
# ── Histograma 3: Años de experiencia laboral total ──────────────────────────
df_wexp = df.dropna(subset=["WorkExp"])

fig, ax = plt.subplots(figsize=(12, 5))

sns.histplot(
    data=df_wexp, x="WorkExp",
    bins=40, kde=True, color="mediumpurple", alpha=0.75, ax=ax
)

med_wexp = df_wexp["WorkExp"].median()
ax.axvline(med_wexp, color="crimson", ls="--", lw=2, label=f"Mediana: {med_wexp:.0f} años")

ax.set_title("Distribución de Años de Experiencia Laboral Total", fontsize=15, fontweight="bold")
ax.set_xlabel("Años de experiencia laboral")
ax.set_ylabel("Frecuencia")
ax.legend()
plt.tight_layout()
plt.savefig("hist_exp_laboral.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mediana: {med_wexp:.1f} años  |  Media: {df_wexp['WorkExp'].mean():.1f} años")

---
### 4.3 Valores atípicos — Boxplots

> Los boxplots se construyen **sin flyers** (outliers no se grafican) para mejorar
  la legibilidad, pero los valores atípicos sí son reportados numéricamente.

In [ ]:
# ── Boxplot 1: Salario por modalidad de trabajo ───────────────────────────────
REMOTE_ORDER  = ["In-person", "Hybrid (some remote, some in-person)", "Remote"]
REMOTE_LABELS = {"In-person": "Presencial",
                 "Hybrid (some remote, some in-person)": "Híbrido",
                 "Remote": "Remoto"}

df_bp1 = df_sal[df_sal["RemoteWork"].isin(REMOTE_ORDER)].copy()
df_bp1["Modalidad"] = df_bp1["RemoteWork"].map(REMOTE_LABELS)

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=df_bp1, x="Modalidad", y="ConvertedCompYearly",
    order=["Presencial", "Híbrido", "Remoto"],
    palette="Set2", showfliers=False, width=0.5, ax=ax
)

ax.set_title("Salario Anual por Modalidad de Trabajo", fontsize=15, fontweight="bold")
ax.set_xlabel("Modalidad de trabajo")
ax.set_ylabel("Salario anual (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${{x/1000:.0f}}K"))
plt.tight_layout()
plt.savefig("box_remotework.png", dpi=150, bbox_inches="tight")
plt.show()

stats_bp1 = df_bp1.groupby("Modalidad")["ConvertedCompYearly"].agg(
    Mediana="median", Media="mean", n="count"
).loc[["Presencial", "Híbrido", "Remoto"]]
stats_bp1["Mediana"] = stats_bp1["Mediana"].apply(lambda x: f"USD {x:,.0f}")
stats_bp1["Media"]   = stats_bp1["Media"].apply(lambda x: f"USD {x:,.0f}")
print(stats_bp1.to_string())

In [ ]:
# ── Boxplot 2: Salario por nivel educativo ────────────────────────────────────
df_bp2 = df_sal[df_sal["EdLevel_short"].isin(EDLEVEL_ORDER)].copy()

fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(
    data=df_bp2, x="EdLevel_short", y="ConvertedCompYearly",
    order=EDLEVEL_ORDER, palette="Blues_d", showfliers=False, ax=ax
)

ax.set_title("Salario Anual por Nivel Educativo", fontsize=15, fontweight="bold")
ax.set_xlabel("Nivel educativo (orden creciente →)")
ax.set_ylabel("Salario anual (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${{x/1000:.0f}}K"))
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("box_edlevel.png", dpi=150, bbox_inches="tight")
plt.show()

stats_bp2 = df_bp2.groupby("EdLevel_short", observed=True)["ConvertedCompYearly"].median()
stats_bp2 = stats_bp2.reindex(EDLEVEL_ORDER)
stats_bp2 = stats_bp2.apply(lambda x: f"USD {x:,.0f}" if pd.notna(x) else "N/D")
print("Salario mediano por nivel educativo:")
print(stats_bp2.to_string())

In [ ]:
# ── Boxplot 3: Salario por tamaño de empresa ─────────────────────────────────
df_bp3 = df_sal[df_sal["OrgSize_short"].isin(ORGSIZE_ORDER)].copy()

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(
    data=df_bp3, x="OrgSize_short", y="ConvertedCompYearly",
    order=ORGSIZE_ORDER, palette="Oranges", showfliers=False, ax=ax
)

ax.set_title("Salario Anual por Tamaño de Empresa", fontsize=15, fontweight="bold")
ax.set_xlabel("Tamaño de la empresa (empleados)")
ax.set_ylabel("Salario anual (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${{x/1000:.0f}}K"))
plt.tight_layout()
plt.savefig("box_orgsize.png", dpi=150, bbox_inches="tight")
plt.show()

# Detección de outliers con IQR
Q1, Q3 = df_bp3["ConvertedCompYearly"].quantile([0.25, 0.75])
IQR    = Q3 - Q1
n_out  = ((df_bp3["ConvertedCompYearly"] < Q1 - 1.5*IQR) |
           (df_bp3["ConvertedCompYearly"] > Q3 + 1.5*IQR)).sum()
print(f"Outliers salariales (método IQR) en subconjunto df_bp3: {n_out:,}  "
      f"({n_out/len(df_bp3)*100:.1f} %)")

---
### 4.4 Relación entre variables — Scatterplots

> Se usa una muestra aleatoria de 5 000 puntos para evitar sobreploteo,
  junto con una línea de tendencia lineal y el coeficiente de correlación de Pearson.

In [ ]:
# ── Scatterplot 1: Experiencia profesional vs. Salario ───────────────────────
df_sc1 = df_sal.dropna(subset=["YearsCodePro"]).sample(n=min(5_000, len(df_sal)), random_state=42)

fig, ax = plt.subplots(figsize=(12, 7))

sc = ax.scatter(
    df_sc1["YearsCodePro"], df_sc1["ConvertedCompYearly"],
    c=df_sc1["ConvertedCompYearly"], cmap="viridis",
    alpha=0.35, s=18, edgecolors="none"
)

# Línea de tendencia
m, b = np.polyfit(df_sc1["YearsCodePro"], df_sc1["ConvertedCompYearly"], 1)
xs   = np.linspace(df_sc1["YearsCodePro"].min(), df_sc1["YearsCodePro"].max(), 200)
ax.plot(xs, m*xs + b, color="crimson", lw=2.5, label="Tendencia lineal")

plt.colorbar(sc, ax=ax, label="Salario anual (USD)")
ax.set_title("Años de Experiencia Profesional vs. Salario Anual", fontsize=15, fontweight="bold")
ax.set_xlabel("Años de experiencia profesional")
ax.set_ylabel("Salario anual (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${{x/1000:.0f}}K"))

r = df_sal.dropna(subset=["YearsCodePro"])[["YearsCodePro", "ConvertedCompYearly"]].corr().iloc[0, 1]
ax.text(0.05, 0.93, f"Pearson r = {r:.3f}", transform=ax.transAxes,
        fontsize=13, bbox=dict(boxstyle="round", fc="white", alpha=0.85))
ax.legend()
plt.tight_layout()
plt.savefig("scatter_exp_sal.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Correlación de Pearson (YearsCodePro vs Salario): {r:.3f}")
print("→ Correlación positiva moderada: a mayor experiencia, mayor salario.")

In [ ]:
# ── Scatterplot 2: Experiencia laboral total vs. Salario ─────────────────────
df_sc2 = df_sal.dropna(subset=["WorkExp"]).sample(n=min(5_000, len(df_sal)), random_state=42)

fig, ax = plt.subplots(figsize=(12, 7))

sc2 = ax.scatter(
    df_sc2["WorkExp"], df_sc2["ConvertedCompYearly"],
    c=df_sc2["ConvertedCompYearly"], cmap="plasma",
    alpha=0.35, s=18, edgecolors="none"
)

m2, b2 = np.polyfit(df_sc2["WorkExp"], df_sc2["ConvertedCompYearly"], 1)
xs2    = np.linspace(df_sc2["WorkExp"].min(), df_sc2["WorkExp"].max(), 200)
ax.plot(xs2, m2*xs2 + b2, color="crimson", lw=2.5, label="Tendencia lineal")

plt.colorbar(sc2, ax=ax, label="Salario anual (USD)")
ax.set_title("Experiencia Laboral Total vs. Salario Anual", fontsize=15, fontweight="bold")
ax.set_xlabel("Años de experiencia laboral total")
ax.set_ylabel("Salario anual (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${{x/1000:.0f}}K"))

r2 = df_sal.dropna(subset=["WorkExp"])[["WorkExp", "ConvertedCompYearly"]].corr().iloc[0, 1]
ax.text(0.05, 0.93, f"Pearson r = {r2:.3f}", transform=ax.transAxes,
        fontsize=13, bbox=dict(boxstyle="round", fc="white", alpha=0.85))
ax.legend()
plt.tight_layout()
plt.savefig("scatter_wexp_sal.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Correlación de Pearson (WorkExp vs Salario): {r2:.3f}")

---
### 4.5 Visualizaciones adicionales

#### A1 — Top 10 países por cantidad de respondentes

In [ ]:
# ── Adicional 1: Barras horizontales — Top 10 países ─────────────────────────
top10 = df["Country"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(11, 6))
colors = sns.color_palette("muted", 10)
bars   = ax.barh(top10.index[::-1], top10.values[::-1], color=colors[::-1], edgecolor="white")

for bar, val in zip(bars, top10.values[::-1]):
    ax.text(bar.get_width() + 150, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", ha="left", fontsize=11)

ax.set_title("Top 10 Países por Cantidad de Respondentes", fontsize=15, fontweight="bold")
ax.set_xlabel("Cantidad de respondentes")
ax.set_xlim(0, top10.max() * 1.15)
plt.tight_layout()
plt.savefig("bar_paises.png", dpi=150, bbox_inches="tight")
plt.show()

pct_us = top10["United States of America"] / len(df) * 100
print(f"EE.UU. representa el {pct_us:.1f} % de las respuestas.")

#### A2 — Salario mediano por tipo de desarrollador

In [ ]:
# ── Adicional 2: Barras — Salario mediano por rol ─────────────────────────────
devtype_stats = (
    df_sal[df_sal["DevType_simp"].notna()]
    .groupby("DevType_simp")["ConvertedCompYearly"]
    .agg(Mediana="median", n="count")
    .query("n >= 80")                        # Solo roles con suficientes respuestas
    .sort_values("Mediana", ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 7))
palette = sns.color_palette("RdYlGn", len(devtype_stats))
bars    = ax.barh(devtype_stats.index, devtype_stats["Mediana"],
                  color=palette, edgecolor="white")

for bar, val in zip(bars, devtype_stats["Mediana"]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f"${{val/1000:.0f}}K", va="center", ha="left", fontsize=11)

ax.set_title("Salario Mediano Anual por Tipo de Desarrollador", fontsize=15, fontweight="bold")
ax.set_xlabel("Salario mediano anual (USD)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${{x/1000:.0f}}K"))
ax.set_xlim(0, devtype_stats["Mediana"].max() * 1.18)
plt.tight_layout()
plt.savefig("bar_devtype_sal.png", dpi=150, bbox_inches="tight")
plt.show()

#### A3 — Salario mediano según tramos de experiencia profesional

In [ ]:
# ── Adicional 3: Línea — Salario mediano por tramo de experiencia ─────────────
df_tramos = df_sal.dropna(subset=["YearsCodePro"]).copy()
bins_exp   = [0, 1, 3, 5, 8, 12, 17, 25, 50]
labels_exp = ["<1", "1-3", "3-5", "5-8", "8-12", "12-17", "17-25", ">25"]
df_tramos["Tramo_exp"] = pd.cut(df_tramos["YearsCodePro"], bins=bins_exp, labels=labels_exp)

evolucion = df_tramos.groupby("Tramo_exp", observed=True)["ConvertedCompYearly"].agg(
    mediana="median", q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75), n="count"
).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))

ax.fill_between(range(len(evolucion)), evolucion["q25"], evolucion["q75"],
                alpha=0.25, color="steelblue", label="Rango IQR (P25–P75)")
ax.plot(range(len(evolucion)), evolucion["mediana"],
        marker="o", color="steelblue", lw=2.5, markersize=7, label="Mediana")

ax.set_xticks(range(len(evolucion)))
ax.set_xticklabels(evolucion["Tramo_exp"].astype(str))
ax.set_title("Evolución del Salario Mediano según Tramos de Experiencia Profesional",
             fontsize=15, fontweight="bold")
ax.set_xlabel("Años de experiencia profesional")
ax.set_ylabel("Salario anual (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${{x/1000:.0f}}K"))
ax.legend()
plt.tight_layout()
plt.savefig("linea_exp_sal.png", dpi=150, bbox_inches="tight")
plt.show()

print("Salario mediano y cantidad de respuestas por tramo:")
evolucion["mediana"] = evolucion["mediana"].apply(lambda x: f"USD {x:,.0f}")
print(evolucion[["Tramo_exp", "mediana", "n"]].to_string(index=False))

---
### 4.6 Correlaciones — Mapa de calor

In [ ]:
# ── Heatmap de correlación ────────────────────────────────────────────────────
COLS_CORR = ["ConvertedCompYearly", "YearsCodePro", "WorkExp"]
LABELS_CORR = {
    "ConvertedCompYearly": "Salario anual",
    "YearsCodePro":        "Exp. profesional",
    "WorkExp":             "Exp. laboral",
}

corr_matrix = df_sal[COLS_CORR].dropna().corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr_matrix, annot=True, fmt=".3f",
    cmap="coolwarm", vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    xticklabels=[LABELS_CORR[c] for c in COLS_CORR],
    yticklabels=[LABELS_CORR[c] for c in COLS_CORR],
    ax=ax
)
ax.set_title("Matriz de Correlación (Pearson) — Variables Numéricas",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("heatmap_corr.png", dpi=150, bbox_inches="tight")
plt.show()

print("Correlación más fuerte: experiencia laboral total ↔ experiencia profesional")
print("Ambas variables de experiencia tienen correlación positiva moderada con el salario.")

---
## 5. Respuesta a las hipótesis

Cada hipótesis se responde con métricas concretas y el gráfico de referencia.

### H1 — Experiencia profesional y salario

In [ ]:
# ── H1: ¿Mayor experiencia → mayor salario? ──────────────────────────────────
df_h1 = df_sal.dropna(subset=["YearsCodePro"]).copy()
df_h1["Tramo"] = pd.cut(df_h1["YearsCodePro"],
                         bins=[0, 2, 5, 10, 20, 50],
                         labels=["0-2 años", "3-5 años", "6-10 años", "11-20 años", ">20 años"])

tabla_h1 = df_h1.groupby("Tramo", observed=True)["ConvertedCompYearly"].agg(
    Mediana="median", Media="mean", n="count"
)
tabla_h1["Mediana"] = tabla_h1["Mediana"].apply(lambda x: f"USD {x:,.0f}")
tabla_h1["Media"]   = tabla_h1["Media"].apply(lambda x: f"USD {x:,.0f}")

r_h1 = df_sal.dropna(subset=["YearsCodePro"])[["YearsCodePro","ConvertedCompYearly"]].corr().iloc[0,1]

print("Salario por tramo de experiencia profesional:")
print(tabla_h1.to_string())
print(f"\nCorrelación de Pearson: {r_h1:.3f}")
print()
print("✅ H1 CONFIRMADA — A mayor experiencia profesional, mayor salario mediano.")
print("   El incremento es sistemático en todos los tramos y la correlación es positiva.")

### H2 — Modalidad de trabajo y salario

In [ ]:
# ── H2: ¿Remoto → mayor salario? ─────────────────────────────────────────────
RMAP = {"In-person": "Presencial",
        "Hybrid (some remote, some in-person)": "Híbrido",
        "Remote": "Remoto"}

df_h2 = df_sal[df_sal["RemoteWork"].isin(RMAP)].copy()
df_h2["Modalidad"] = df_h2["RemoteWork"].map(RMAP)

tabla_h2 = df_h2.groupby("Modalidad")["ConvertedCompYearly"].agg(
    Mediana="median", Media="mean", n="count"
).loc[["Presencial", "Híbrido", "Remoto"]]
tabla_h2["Mediana"] = tabla_h2["Mediana"].apply(lambda x: f"USD {x:,.0f}")
tabla_h2["Media"]   = tabla_h2["Media"].apply(lambda x: f"USD {x:,.0f}")

print("Salario por modalidad de trabajo:")
print(tabla_h2.to_string())
print()
print("⚠️  H2 PARCIALMENTE CONFIRMADA — El trabajo remoto no siempre supera al presencial.")
print("   El salario mediano de Híbrido suele superar al Remoto puro, posiblemente porque")
print("   muchas empresas grandes (que pagan más) exigen cierta presencialidad.")

### H3 — Nivel educativo y salario

In [ ]:
# ── H3: ¿Mayor educación → mayor salario? ────────────────────────────────────
df_h3 = df_sal[df_sal["EdLevel_short"].isin(EDLEVEL_ORDER)].copy()

tabla_h3 = df_h3.groupby("EdLevel_short", observed=True)["ConvertedCompYearly"].agg(
    Mediana="median", n="count"
).reindex(EDLEVEL_ORDER)
tabla_h3["Mediana"] = tabla_h3["Mediana"].apply(lambda x: f"USD {x:,.0f}" if pd.notna(x) else "N/D")

print("Salario mediano por nivel educativo (orden creciente):")
print(tabla_h3.to_string())
print()
print("⚠️  H3 PARCIALMENTE CONFIRMADA — La relación no es perfectamente monotónica.")
print("   Licenciatura y Maestría muestran salarios altos, pero 'Univ. sin título' a veces")
print("   supera a Tecnicatura, lo que sugiere que la experiencia práctica pesa tanto")
print("   o más que el título formal en la industria del software.")

### H4 — Tamaño de empresa y salario

In [ ]:
# ── H4: ¿Empresa más grande → mayor salario? ─────────────────────────────────
df_h4 = df_sal[df_sal["OrgSize_short"].isin(ORGSIZE_ORDER)].copy()

tabla_h4 = df_h4.groupby("OrgSize_short", observed=True)["ConvertedCompYearly"].agg(
    Mediana="median", n="count"
).reindex(ORGSIZE_ORDER)
tabla_h4["Mediana"] = tabla_h4["Mediana"].apply(lambda x: f"USD {x:,.0f}")

print("Salario mediano por tamaño de empresa:")
print(tabla_h4.to_string())
print()
print("✅ H4 CONFIRMADA — En términos generales, las empresas más grandes pagan más.")
print("   Las empresas de +10K empleados tienen la mediana salarial más alta.")
print("   Excepción: empresas muy pequeñas (solo o 2-9) a veces tienen outliers altos")
print("   por freelancers o fundadores con ingresos elevados.")

### H5 — Tipo de desarrollador y salario

In [ ]:
# ── H5: ¿El rol determina diferencias salariales? ────────────────────────────
df_h5 = df_sal[df_sal["DevType_simp"].notna()].copy()

tabla_h5 = df_h5.groupby("DevType_simp")["ConvertedCompYearly"].agg(
    Mediana="median", n="count"
).query("n >= 80").sort_values("Mediana", ascending=False)
tabla_h5["Mediana"] = tabla_h5["Mediana"].apply(lambda x: f"USD {x:,.0f}")

print("Salario mediano por tipo de desarrollador (ranking descendente):")
print(tabla_h5.to_string())
print()
print("✅ H5 CONFIRMADA — Existen diferencias salariales significativas entre roles.")
print("   Engineering Manager y DevOps/SRE lideran el ranking salarial.")
print("   Data Scientist/ML también aparece en los primeros puestos.")
print("   Front-end y Game Developer se ubican en los niveles más bajos.")

---
## 6. Conclusiones finales

### Hallazgos principales

A partir del análisis exploratorio de ~89 000 respuestas de desarrolladores globales
en la encuesta de Stack Overflow 2023, se obtuvieron los siguientes hallazgos:

1. **Distribución salarial sesgada** — Los salarios tienen una distribución con fuerte
   asimetría positiva. La mayoría de los desarrolladores gana menos de USD 100K/año,
   pero un segmento reduce la representatividad de la media como medida central;
   la **mediana** es el estadístico más adecuado para describir el ingreso típico.

2. **La experiencia profesional importa** *(H1 ✅)* — La correlación positiva entre
   años de experiencia y salario es clara y consistente en todos los tramos analizados.
   Pasar de 0-2 a 3-5 años de experiencia ya representa un salto salarial relevante.

3. **Trabajo remoto no garantiza mayor pago** *(H2 ⚠️)* — Contrariamente a la creencia
   popular, el trabajo híbrido muestra medianas salariales levemente superiores al
   remoto puro. Esto sugiere que las grandes empresas, que tienden a pagar más,
   exigen aún cierta presencialidad.

4. **Educación formal no es determinante absoluto** *(H3 ⚠️)* — Si bien los niveles
   altos (Maestría, Doctorado) se asocian a salarios mayores, la diferencia con
   Licenciatura es moderada. Desarrolladores sin título universitario completo pueden
   superar en ingresos a quienes tienen tecnicatura, evidenciando que la experiencia
   práctica es muy valorada en la industria tech.

5. **Empresas grandes pagan más** *(H4 ✅)* — La relación entre tamaño organizacional
   y salario es positiva y robusta. Las empresas de más de 10 000 empleados presentan
   la mediana salarial más alta del conjunto.

6. **El rol define el salario** *(H5 ✅)* — Engineering Manager, DevOps/SRE y
   Data Scientist lideran el ranking. La diferencia entre el rol mejor y peor pagado
   puede superar los USD 40 000 anuales en mediana.

### Limitaciones del análisis

- Los salarios están reportados en USD ajustados por PPP, pero la muestra está
  sesgada hacia EE.UU. (> 20 % de las respuestas), lo que infla los valores globales.
- El ~70 % de los encuestados no reportó su salario, lo cual puede introducir sesgo
  de no respuesta (quienes responden podrían ser los que ganan más o menos).
- `DevType` es multi-valor y la simplificación a un único rol puede no capturar
  perfiles mixtos (ej.: un full-stack que también hace DevOps).